
# Vanilla Zero-shot Baseline — ASAP Essay Sets 1–4

Notebook này tạo baseline **Vanilla Mistral 7B**: chỉ gọi model chấm điểm bài essay, không train/supervised learning.

Thiết kế theo yêu cầu:
- Dùng 3 file `asap_train`, `asap_val`, `asap_test`.
- `concat` cả 3 file lại rồi trộn.
- Chỉ lấy **10%** làm evaluation zero-shot.
- Chỉ chạy **Prompt/Essay Set 1–4**.
- Tính metric chính: **Quadratic Weighted Kappa (QWK)**.
- Prompt 5–8 bỏ qua vì partner làm.

> Lưu ý: file `train/test/val` trong supervised learning ở đây được gộp lại vì baseline zero-shot không học trên các file đó.


In [1]:

# Nếu chạy trên Colab/Kaggle và thiếu package, mở comment dòng dưới.
# !pip install -q transformers accelerate bitsandbytes scikit-learn pandas tqdm
!pip install -U "bitsandbytes>=0.46.1" accelerate transformers
import os
import re
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 120.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0



## 1. Cấu hình

Sửa `DATA_DIR` cho đúng nơi chứa 3 file dataset của bạn.

Notebook sẽ tự tìm file có tên gần giống:
- `asap_train.csv`, `train.csv`
- `asap_val.csv`, `val.csv`, `dev.csv`
- `asap_test.csv`, `test.csv`

Bạn cũng có thể sửa trực tiếp `TRAIN_PATH`, `VAL_PATH`, `TEST_PATH`.


In [2]:

# ====== CẤU HÌNH ĐƯỜNG DẪN ======
DATA_DIR = Path("./dataset")  # sửa nếu cần, ví dụ: Path("/kaggle/input/your-dataset")

TRAIN_PATH = Path("/content/asap_train.csv")
VAL_PATH = Path("/content/asap_val.csv")
TEST_PATH = Path("/content/asap_test.csv")

# Chỉ chạy prompt 1-4
ESSAY_SETS_TO_RUN = [1, 2, 3, 4]

# Lấy 10% data sau khi concat train+val+test
EVAL_FRAC = 0.10
RANDOM_SEED = 222

# Nếu muốn test nhanh trước, đặt số nhỏ, ví dụ 30.
# Nếu muốn chạy đủ 10%, để None.
MAX_EVAL_PER_SET = None

# Model vanilla baseline
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Generation config
MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
DO_SAMPLE = False

# Lưu kết quả
OUTPUT_DIR = Path("./vanilla_baseline_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:

def find_first_existing(data_dir: Path, candidates):
    for name in candidates:
        p = data_dir / name
        if p.exists():
            return p
    return None

if TRAIN_PATH is None:
    TRAIN_PATH = find_first_existing(DATA_DIR, [
        "asap_train.csv", "train.csv", "training_set_rel3.csv", "asap_train.tsv", "train.tsv"
    ])
if VAL_PATH is None:
    VAL_PATH = find_first_existing(DATA_DIR, [
        "asap_val.csv", "asap_validation.csv", "val.csv", "valid.csv", "dev.csv", "asap_val.tsv", "val.tsv"
    ])
if TEST_PATH is None:
    TEST_PATH = find_first_existing(DATA_DIR, [
        "asap_test.csv", "test.csv", "asap_test.tsv", "test.tsv"
    ])

print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:  ", VAL_PATH)
print("TEST_PATH: ", TEST_PATH)

missing = [name for name, p in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)] if p is None or not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"Không tìm thấy file: {missing}. Hãy sửa DATA_DIR hoặc TRAIN_PATH/VAL_PATH/TEST_PATH."
    )


TRAIN_PATH: /content/asap_train.csv
VAL_PATH:   /content/asap_val.csv
TEST_PATH:  /content/asap_test.csv



## 2. Load và concat 3 split

Notebook cố gắng tự nhận dạng cột:
- Essay set: `essay_set`, `prompt_id`, `set`
- Essay text: `essay`, `full_text`, `text`, `response`
- Score gold:
  - Prompt 1: thường là `domain1_score`, `resolved_score`, hoặc score tổng.
  - Prompt 2: mặc định dùng **Domain 1** (`domain1_score` / `D1_Resolved`) vì bảng bạn đang so P1–P8 thường là 1 score chính mỗi prompt. Nếu bạn muốn chấm trait Domain 2, đổi hàm `pick_gold_column`.
  - Prompt 3–4: thường là `domain1_score`, `resolved_score`, `Resolved CR Score`.


In [4]:

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)

train_df = read_table(TRAIN_PATH)
val_df = read_table(VAL_PATH)
test_df = read_table(TEST_PATH)

train_df["source_split"] = "train"
val_df["source_split"] = "val"
test_df["source_split"] = "test"

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("Shape train:", train_df.shape)
print("Shape val:  ", val_df.shape)
print("Shape test: ", test_df.shape)
print("Shape all:  ", all_df.shape)
print("\nColumns:")
print(all_df.columns.tolist())
all_df.head()


Shape train: (9084, 6)
Shape val:   (1296, 6)
Shape test:  (2596, 6)
Shape all:   (12976, 6)

Columns:
['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm', 'source_split']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm,source_split
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75,train
1,9985,4,The author concluded this sentence because he ...,0.0,0.00,train
2,3231,2,"I can think of several books that, I would not...",4.0,0.60,train
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65,train
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77,train


In [5]:

def find_col(df, candidates, required=True):
    normalized = {c.lower().strip().replace(" ", "_"): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    # fallback contains
    for cand in candidates:
        key = cand.lower().strip().replace(" ", "_")
        for norm, original in normalized.items():
            if key in norm:
                return original
    if required:
        raise KeyError(f"Không tìm thấy cột trong candidates={candidates}. Columns={df.columns.tolist()}")
    return None

ESSAY_SET_COL = find_col(all_df, ["essay_set", "prompt_id", "set", "essayset"])
ESSAY_COL = find_col(all_df, ["essay", "full_text", "text", "response", "essay_text"])

def pick_gold_column(df, essay_set_id):
    """
    Chọn gold score cho từng essay set.
    Mặc định:
    - P1: resolved/domain1/tổng score.
    - P2: Domain 1 score, vì ASAP set 2 có 2 trait; bảng baseline thường dùng score chính D1.
    - P3/P4: resolved/domain1 CR score.
    """
    if essay_set_id == 2:
        candidates = [
            "domain1_score", "D1_Resolved", "d1_resolved", "resolved_score",
            "Resolved Score", "score", "final_score"
        ]
    else:
        candidates = [
            "domain1_score", "resolved_score", "Resolved Score", "Resolved CR Score",
            "score", "final_score"
        ]
    return find_col(df, candidates, required=True)

print("ESSAY_SET_COL:", ESSAY_SET_COL)
print("ESSAY_COL:    ", ESSAY_COL)

for s in ESSAY_SETS_TO_RUN:
    sub = all_df[all_df[ESSAY_SET_COL].astype(int) == s]
    print(f"P{s}: rows={len(sub)}, gold_col={pick_gold_column(sub, s) if len(sub) else 'N/A'}")


ESSAY_SET_COL: essay_set
ESSAY_COL:     essay
P1: rows=1783, gold_col=domain1_score
P2: rows=1800, gold_col=domain1_score
P3: rows=1726, gold_col=domain1_score
P4: rows=1770, gold_col=domain1_score



## 3. Prompt text và score range cho Essay Set 1–4

Score range:
- P1: `2–12`
- P2: `1–6` cho Domain 1
- P3: `0–3`
- P4: `0–3`


In [6]:

PROMPT_INFO = {
    1: {
        "score_min": 2,
        "score_max": 12,
        "prompt": """More and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends.

Write a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.""",
        "rubric_short": "Score 2-12. Higher scores mean a clearer position, stronger support, better organization, fluency, transitions, and awareness of audience."
    },
    2: {
        "score_min": 1,
        "score_max": 6,
        "prompt": """Censorship in the Libraries

"All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us." --Katherine Paterson, Author

Write a persuasive essay to a newspaper reflecting your views on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.""",
        "rubric_short": "Domain 1 score 1-6. Higher scores mean stronger task accomplishment, relevant and developed ideas, organization, style, voice, and audience awareness."
    },
    3: {
        "score_min": 0,
        "score_max": 3,
        "prompt": """Read the source essay "ROUGH ROAD AHEAD: Do Not Exceed Posted Speed Limit" by Joe Kurmaskie.

Write a response that explains how the features of the setting affect the cyclist. In your response, include examples from the essay that support your conclusion.""",
        "rubric_short": "Score 0-3. Higher scores demonstrate stronger understanding of the text, use of explicit and implied information, and explanation beyond literal meaning."
    },
    4: {
        "score_min": 0,
        "score_max": 3,
        "prompt": """Read the source essay "Winter Hibiscus" by Minfong Ho.

Read the last paragraph of the story:

"When they come back, Saeng vowed silently to herself, in the spring, when the snows melt and the geese return and this hibiscus is budding, then I will take that test again."

Write a response that explains why the author concludes the story with this paragraph. In your response, include details and examples from the story that support your ideas.""",
        "rubric_short": "Score 0-3. Higher scores demonstrate stronger understanding of the text, use of explicit and implied information, and explanation beyond literal meaning."
    },
}



## 4. Tạo evaluation set 10%

Cách lấy mẫu:
1. Lọc prompt 1–4.
2. Trộn toàn bộ dữ liệu đã concat.
3. Lấy 10% cho evaluation.
4. Có stratify theo `essay_set` để mỗi prompt đều có dữ liệu.

Nếu muốn mỗi prompt chỉ lấy tối đa vài bài để test pipeline, đặt `MAX_EVAL_PER_SET`.


In [7]:

rng = np.random.default_rng(RANDOM_SEED)

df_14 = all_df[all_df[ESSAY_SET_COL].astype(int).isin(ESSAY_SETS_TO_RUN)].copy()
df_14[ESSAY_SET_COL] = df_14[ESSAY_SET_COL].astype(int)

eval_parts = []
for s in ESSAY_SETS_TO_RUN:
    sub = df_14[df_14[ESSAY_SET_COL] == s].copy()
    n = max(1, int(round(len(sub) * EVAL_FRAC)))
    part = sub.sample(n=n, random_state=RANDOM_SEED)
    if MAX_EVAL_PER_SET is not None:
        part = part.sample(n=min(MAX_EVAL_PER_SET, len(part)), random_state=RANDOM_SEED)
    eval_parts.append(part)

eval_df = pd.concat(eval_parts, ignore_index=True)
eval_df = eval_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print("Total P1-P4 rows:", len(df_14))
print("Eval rows:", len(eval_df))
print(eval_df[ESSAY_SET_COL].value_counts().sort_index())
eval_df[[ESSAY_SET_COL, ESSAY_COL, "source_split"]].head()


Total P1-P4 rows: 7079
Eval rows: 708
essay_set
1    178
2    180
3    173
4    177
Name: count, dtype: int64


,essay_set,essay,source_split
0,3,The setting effects the cyclist very much. By ...,train
1,3,All the features of the environment through wh...,train
2,2,In this persuation essay it will talk about th...,train
3,1,"@CAPS1 dinners ready okay mom, one second' cli...",test
4,4,The author concludes this story with the few w...,train



## 5. Load Vanilla Mistral 7B

Nếu GPU yếu, notebook dùng 4-bit quantization qua `bitsandbytes`.

Nếu bạn chạy local không có GPU, Mistral 7B sẽ rất chậm. Nên chạy trên Kaggle/Colab GPU.


In [8]:

def load_llm(model_name=MODEL_NAME):
    use_cuda = torch.cuda.is_available()
    print("CUDA available:", use_cuda)
    if use_cuda:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            device_map="cpu",
            trust_remote_code=True,
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )
    return gen, tokenizer

gen, tokenizer = load_llm()


CUDA available: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


## 6. Hàm build prompt và parse điểm

Vanilla zero-shot nghĩa là:
- Không đưa example.
- Không retrieve quote.
- Không fine-tune.
- Chỉ đưa prompt, rubric ngắn, essay, yêu cầu trả về 1 số.


In [9]:

def build_scoring_prompt(essay_set_id, essay_text):
    info = PROMPT_INFO[int(essay_set_id)]
    lo, hi = info["score_min"], info["score_max"]

    user_msg = f"""You are an expert essay scorer.

Essay prompt:
{info["prompt"]}

Rubric summary:
{info["rubric_short"]}

Task:
Score the student's essay with an integer from {lo} to {hi}.
Return only the integer score. Do not explain.

Student essay:
{essay_text}
"""

    # Mistral Instruct format
    return f"<s>[INST] {user_msg.strip()} [/INST]"

def parse_score(raw_text, essay_set_id):
    info = PROMPT_INFO[int(essay_set_id)]
    lo, hi = info["score_min"], info["score_max"]

    # Lấy phần model sinh sau [/INST] nếu có
    text = raw_text.split("[/INST]")[-1].strip()

    # Ưu tiên số nguyên đầu tiên
    nums = re.findall(r"-?\d+", text)
    if not nums:
        return None

    val = int(nums[0])
    val = max(lo, min(hi, val))
    return val

def score_one_essay(essay_set_id, essay_text):
    prompt = build_scoring_prompt(essay_set_id, essay_text)
    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.eos_token_id,
        "return_full_text": True,
    }
    if DO_SAMPLE:
        gen_kwargs["temperature"] = TEMPERATURE

    out = gen(prompt, **gen_kwargs)[0]["generated_text"]
    pred = parse_score(out, essay_set_id)
    return pred, out



## 7. Chạy inference

Cell này có thể lâu nếu evaluation set lớn. Kết quả được lưu checkpoint CSV sau mỗi vài bài.


In [10]:

pred_rows = []
checkpoint_path = OUTPUT_DIR / "vanilla_mistral_p1_p4_eval_predictions_checkpoint.csv"

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    essay_set_id = int(row[ESSAY_SET_COL])
    essay_text = str(row[ESSAY_COL])

    gold_col = pick_gold_column(eval_df[eval_df[ESSAY_SET_COL] == essay_set_id], essay_set_id)
    gold = row[gold_col]

    try:
        pred, raw = score_one_essay(essay_set_id, essay_text)
    except Exception as e:
        pred, raw = None, f"ERROR: {repr(e)}"

    pred_rows.append({
        "row_id": i,
        "essay_set": essay_set_id,
        "source_split": row.get("source_split", None),
        "gold_col": gold_col,
        "gold": gold,
        "pred": pred,
        "raw_generation": raw,
        "essay": essay_text,
    })

    if (i + 1) % 10 == 0:
        pd.DataFrame(pred_rows).to_csv(checkpoint_path, index=False)

pred_df = pd.DataFrame(pred_rows)
pred_path = OUTPUT_DIR / "vanilla_mistral_p1_p4_eval_predictions.csv"
pred_df.to_csv(pred_path, index=False)

print("Saved:", pred_path)
pred_df.head()


  0%|          | 0/708 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_

Saved: vanilla_baseline_outputs/vanilla_mistral_p1_p4_eval_predictions.csv


,row_id,essay_set,source_split,gold_col,gold,pred,raw_generation,essay
0,0,3,train,domain1_score,2.0,1.0,<s>[INST] You are an expert essay scorer.\n\nE...,The setting effects the cyclist very much. By ...
1,1,3,train,domain1_score,2.0,3.0,<s>[INST] You are an expert essay scorer.\n\nE...,All the features of the environment through wh...
2,2,2,train,domain1_score,4.0,5.0,<s>[INST] You are an expert essay scorer.\n\nE...,In this persuation essay it will talk about th...
3,3,1,test,domain1_score,9.0,2.0,<s>[INST] You are an expert essay scorer.\n\nE...,"@CAPS1 dinners ready okay mom, one second' cli..."
4,4,4,train,domain1_score,1.0,3.0,<s>[INST] You are an expert essay scorer.\n\nE...,The author concludes this story with the few w...



## 8. Tính QWK theo từng prompt và average

QWK dùng `sklearn.metrics.cohen_kappa_score(weights="quadratic")`.

Average trong bảng thường lấy trung bình macro qua P1–P4 nếu chỉ chạy 1–4.


In [11]:

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

metric_rows = []
for s in ESSAY_SETS_TO_RUN:
    sub = pred_df[(pred_df["essay_set"] == s) & pred_df["pred"].notna() & pred_df["gold"].notna()].copy()
    sub["gold"] = sub["gold"].astype(int)
    sub["pred"] = sub["pred"].astype(int)

    if len(sub) == 0:
        score = np.nan
    else:
        score = qwk(sub["gold"], sub["pred"])

    metric_rows.append({
        "method": "Vanilla Mistral 7B zero-shot",
        "essay_set": f"P{s}",
        "n": len(sub),
        "qwk": score,
        "gold_min": sub["gold"].min() if len(sub) else np.nan,
        "gold_max": sub["gold"].max() if len(sub) else np.nan,
        "pred_min": sub["pred"].min() if len(sub) else np.nan,
        "pred_max": sub["pred"].max() if len(sub) else np.nan,
    })

metrics_df = pd.DataFrame(metric_rows)
avg_qwk = metrics_df["qwk"].mean(skipna=True)

display(metrics_df)
print("Macro Avg QWK P1-P4:", avg_qwk)

metrics_path = OUTPUT_DIR / "vanilla_mistral_p1_p4_qwk_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)


,method,essay_set,n,qwk,gold_min,gold_max,pred_min,pred_max
0,Vanilla Mistral 7B zero-shot,P1,178,0.482444,2,12,2,11
1,Vanilla Mistral 7B zero-shot,P2,180,0.317051,1,6,1,5
2,Vanilla Mistral 7B zero-shot,P3,172,0.625643,0,3,0,3
3,Vanilla Mistral 7B zero-shot,P4,174,0.548059,0,3,0,3


Macro Avg QWK P1-P4: 0.4932990115558171
Saved: vanilla_baseline_outputs/vanilla_mistral_p1_p4_qwk_metrics.csv


In [12]:

# Format 1 dòng để copy vào bảng
table_row = {f"P{s}": metrics_df.loc[metrics_df["essay_set"] == f"P{s}", "qwk"].iloc[0] for s in ESSAY_SETS_TO_RUN}
table_row["Avg_P1_P4"] = avg_qwk
pd.DataFrame([table_row])


,P1,P2,P3,P4,Avg_P1_P4
0,0.482444,0.317051,0.625643,0.548059,0.493299



## 9. Debug nhanh nếu QWK thấp bất thường

Xem vài case gold/pred/raw generation để kiểm tra model có trả đúng format số không.


In [13]:

debug_cols = ["essay_set", "gold", "pred", "raw_generation", "essay"]
pd.set_option("display.max_colwidth", 300)
pred_df[debug_cols].sample(min(10, len(pred_df)), random_state=RANDOM_SEED)


,essay_set,gold,pred,raw_generation,essay
475,3,3.0,3.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the source essay ""ROUGH ROAD AHEAD: Do Not Exceed Posted Speed Limit"" by Joe Kurmaskie.\n\nWrite a response that explains how the features of the setting affect the cyclist. In your response, include examples from the essay that su...","In the essay, Do Not Exceed Posted Speed Limit by Joe Kurmaskie, the setting greatly affected the cyclist. In this story, the cyclist is heading for Yosemite National Park, thinking it wise, he accepts directions from a group of eccentric old men. He begins his journey confidently on flat terr..."
471,2,2.0,5.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nCensorship in the Libraries\n\n""All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you ...","My thoughts on censorship in libraries is yes some books, music, movies, magazines, and etc., should be taken off of the shelves if they are offensive. If they are left on the shelves some very small child or teens could come up and reach for a book and find that it has bad things in it that are..."
239,4,0.0,1.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the source essay ""Winter Hibiscus"" by Minfong Ho.\n\nRead the last paragraph of the story:\n\n""When they come back, Saeng vowed silently to herself, in the spring, when the snows melt and the geese return and this hibiscus is buddi...",She should and needs too learn that you cant replace a plant with the same thing and expect it too be as equal too it. She needs too learn that lesson in life.
69,4,1.0,3.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the source essay ""Winter Hibiscus"" by Minfong Ho.\n\nRead the last paragraph of the story:\n\n""When they come back, Saeng vowed silently to herself, in the spring, when the snows melt and the geese return and this hibiscus is buddi...",I think the author concluded the story with this paragraph because they wanted to show that she is comited to her nature especially to The hibiscus. She thought that the hibiscus was special because of what she did with it. Like when they said she reached out and touched petal gently. It felt s...
547,3,2.0,3.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the source essay ""ROUGH ROAD AHEAD: Do Not Exceed Posted Speed Limit"" by Joe Kurmaskie.\n\nWrite a response that explains how the features of the setting affect the cyclist. In your response, include examples from the essay that su...",The features of the setting affect the cyclist because they are extreme and do not change. The cyclist was riding on a small road in the desert. In the desert it was very hot and because of this the cyclist was sweating a lot and getting dehydrated. Also because he was in the desert he had no se...
27,1,8.0,6.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nMore and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ...",I disagree what people say about computers. Likewise of corse they teach hand eye coordination. But they are not good enough for people. Lets find out what I have to say to these people who think that computers have a positive effect on people. People are so fat! Wounder why. Because they spend ...
465,2,4.0,5.0,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nCensorship in the Libraries\n\n""All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you ...","I think there should be no books taken off the shelfs. Somethings migh


## 10. Gợi ý báo cáo trong paper

Bạn có thể mô tả baseline này như sau:

> For the vanilla zero-shot baseline, we concatenated the original train, validation, and test splits because no supervised learning was performed. We randomly sampled 10% of the combined data for evaluation, restricted to Essay Sets 1–4. Mistral-7B-Instruct was prompted with the task prompt, a short rubric description, and the student essay, and was asked to output a single integer score within the official score range. Performance was measured using Quadratic Weighted Kappa for each essay set and macro-averaged across Essay Sets 1–4.
